<a href="https://colab.research.google.com/github/Gabriela-Reiss/Algoritmo_Verifica-o_Decolagem/blob/main/Script_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SCRIPT EM PYTHON - ANÁLISE DE PRÉ-DECOLAGEM**

**DADOS USADOS PARA A CONSTRUÇÃO DO SCRIPT:**

Os dados para a construção do script em python foram gerados pelo Gemini em um arquivo de extensão CSV, nomeado: "dataset_telemetria.csv".

Para a execução do script, anexe diretamente o arquivo do dataset na aba "Arquivos" do Notebook ou use o prompt já criado (disponível no documento PDF da atividade) para que o dataset seja criado pela IA


**Carregamento do dataset gerado:**

In [1]:
import pandas as pd

In [2]:
dados = pd.read_csv("dataset_telemetria.csv")

In [3]:
dados.head(10)

,timestamp,temperatura_interna_c,temperatura_externa_c,integridade_estrutural,energia_maxima_kwh,energia_disponivel_kwh,pressao_tanque_bar,status_modulos_criticos
0,2026-09-08 12:00:00,30.52,18.68,1,775.80,367.68,116.10,1
1,2026-09-08 12:15:00,18.45,1.96,1,1029.98,697.80,106.02,1
2,2026-09-08 12:30:00,28.02,23.33,1,901.30,784.54,129.91,1
3,2026-09-08 12:45:00,23.78,0.44,1,1091.44,566.39,210.85,1
4,2026-09-08 13:00:00,28.26,23.25,1,794.30,700.00,75.42,1
5,2026-09-08 13:15:00,24.15,1.73,1,913.91,705.96,133.65,1
6,2026-09-08 13:30:00,34.75,24.94,1,1073.30,593.95,117.67,1
7,2026-09-08 13:45:00,32.18,0.69,1,971.05,133.53,130.09,1
8,2026-09-08 14:00:00,22.16,11.18,1,953.99,905.20,210.57,1
9,2026-09-08 14:15:00,31.19,3.02,1,906.42,491.30,108.39,1


In [4]:
dados.tail()

,timestamp,temperatura_interna_c,temperatura_externa_c,integridade_estrutural,energia_maxima_kwh,energia_disponivel_kwh,pressao_tanque_bar,status_modulos_criticos
25,2026-09-08 18:15:00,28.00,3.05,1,944.04,345.72,126.56,1
26,2026-09-08 18:30:00,21.89,26.69,1,1071.93,354.70,106.90,1
27,2026-09-08 18:45:00,29.37,2.50,1,926.46,886.06,123.55,1
28,2026-09-08 19:00:00,26.04,22.46,1,1061.50,445.00,99.85,1
29,2026-09-08 19:15:00,25.33,9.83,1,993.40,805.54,128.67,1


In [5]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   timestamp                30 non-null     object 
 1   temperatura_interna_c    30 non-null     float64
 2   temperatura_externa_c    30 non-null     float64
 3   integridade_estrutural   30 non-null     int64  
 4   energia_maxima_kwh       30 non-null     float64
 5   energia_disponivel_kwh   30 non-null     float64
 6   pressao_tanque_bar       30 non-null     float64
 7   status_modulos_criticos  30 non-null     int64  
dtypes: float64(5), int64(2), object(1)
memory usage: 2.0+ KB


**Criação de Algoritmo de Classificação dos dados e análise de status para decolagem**

In [6]:
resultados = []

for indice, linha in dados.iterrows():

    temperatura_interna = linha["temperatura_interna_c"]
    temperatura_externa = linha["temperatura_externa_c"]
    integridade_estrutural = linha["integridade_estrutural"]

    energia_maxima = linha["energia_maxima_kwh"]
    energia_disponivel = linha["energia_disponivel_kwh"]

    pressao_tanque = linha["pressao_tanque_bar"]
    status_modulos = linha["status_modulos_criticos"]

    # Cálculo do nível de energia
    nivel_energia = (
        energia_disponivel / energia_maxima
    ) * 100


# definição das condições referente aos valores permitidos em cada variável e mensagem de motivos, caso condição seja falsa
    regras = [
        (
            18 <= temperatura_interna <= 35,
            "temperatura interna fora do limite"
        ),
        (
            -5 <= temperatura_externa <= 30,
            "temperatura externa fora do limite"
        ),
        (
            integridade_estrutural == 1,
            "integridade estrutural comprometida"
        ),
        (
            energia_disponivel <= energia_maxima,
            "energia disponível superior à capacidade máxima"
        ),
        (
            nivel_energia >= 60,
            "nível de energia abaixo de 60%"
        ),
        (
            95 <= pressao_tanque <= 145,
            "pressão do tanque fora do limite"
        ),
        (
            status_modulos == 1,
            "módulo crítico com falha"
        )
    ]



# armazenamento das mensagens na lista de motivos, para uma justificativa de cada decisão da verificação
    motivos = [
        mensagem
        for condicao, mensagem in regras
        if not condicao
    ]


# definição da decisão com base na presença de motivos armazenados (caso tenha motivos = decolagem abortada - estava fora dos limites permitidos)
    decisao = (
        "PRONTO PARA DECOLAR\n"
        if not motivos
        else "DECOLAGEM ABORTADA"
    )

    print(indice,
        linha["timestamp"],
        "-",
        decisao
    )

    if motivos:
        print("  Motivo(s):", "; ".join(motivos), "\n")


# salvando dados de telemetria e decisões em um data frame
    resultados.append({
    "timestamp": linha["timestamp"],
    "temperatura_interna": temperatura_interna,
    "temperatura_externa": temperatura_externa,
    "integridade_estrutural": integridade_estrutural,
    "nivel_energia": nivel_energia,
    "pressao_tanque": pressao_tanque,
    "status_modulos_criticos": status_modulos,
    "decisao": decisao,
    "motivos": "; ".join(motivos) if motivos else "Nenhum"
})

0 2026-09-08 12:00:00 - DECOLAGEM ABORTADA
  Motivo(s): nível de energia abaixo de 60% 

1 2026-09-08 12:15:00 - PRONTO PARA DECOLAR

2 2026-09-08 12:30:00 - PRONTO PARA DECOLAR

3 2026-09-08 12:45:00 - DECOLAGEM ABORTADA
  Motivo(s): nível de energia abaixo de 60%; pressão do tanque fora do limite 

4 2026-09-08 13:00:00 - DECOLAGEM ABORTADA
  Motivo(s): pressão do tanque fora do limite 

5 2026-09-08 13:15:00 - PRONTO PARA DECOLAR

6 2026-09-08 13:30:00 - DECOLAGEM ABORTADA
  Motivo(s): nível de energia abaixo de 60% 

7 2026-09-08 13:45:00 - DECOLAGEM ABORTADA
  Motivo(s): nível de energia abaixo de 60% 

8 2026-09-08 14:00:00 - DECOLAGEM ABORTADA
  Motivo(s): pressão do tanque fora do limite 

9 2026-09-08 14:15:00 - DECOLAGEM ABORTADA
  Motivo(s): nível de energia abaixo de 60% 

10 2026-09-08 14:30:00 - DECOLAGEM ABORTADA
  Motivo(s): nível de energia abaixo de 60% 

11 2026-09-08 14:45:00 - DECOLAGEM ABORTADA
  Motivo(s): nível de energia abaixo de 60% 

12 2026-09-08 15:00:00 -

**Salvando o dataset completo e os resultados em um novo Data Frame**

In [7]:
resultados_df = pd.DataFrame(resultados)

print(resultados_df)

              timestamp  temperatura_interna  temperatura_externa  \
0   2026-09-08 12:00:00                30.52                18.68   
1   2026-09-08 12:15:00                18.45                 1.96   
2   2026-09-08 12:30:00                28.02                23.33   
3   2026-09-08 12:45:00                23.78                 0.44   
4   2026-09-08 13:00:00                28.26                23.25   
5   2026-09-08 13:15:00                24.15                 1.73   
6   2026-09-08 13:30:00                34.75                24.94   
7   2026-09-08 13:45:00                32.18                 0.69   
8   2026-09-08 14:00:00                22.16                11.18   
9   2026-09-08 14:15:00                31.19                 3.02   
10  2026-09-08 14:30:00                21.59                28.00   
11  2026-09-08 14:45:00                24.73                27.01   
12  2026-09-08 15:00:00                27.54                 4.20   
13  2026-09-08 15:15:00           

**Convertendo o Data Frame em JSON para análise da IA**

In [8]:
dados_para_ia = resultados_df.to_json(
    orient="records",
    force_ascii=False
)

## **ANÁLISE ASSISTIDA POR IA**

Visando obter uma análise detalhada dos resultados e dados por IA, utilizamos uma integração com a API Gemini (modelo Gemini 3.6 Flash)

O objetivo dessa integração é permitir que a IA dê sugestões de melhorias e analise os resultados do algoritmo.


In [9]:
import os
from google.colab import userdata

**Para conseguir executar, gere uma API Key no Google IA Studio (https://aistudio.google.com/api-keys) e configure-a na aba "Secrets" do Notebook**

In [10]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [11]:
from google import genai

client = genai.Client()
chat = client.chats.create(model="gemini-3.6-flash")

**Prompt descrevendo as informações que devem ser analisadas pela IA**

In [14]:
# prompt para IA com as informações que desejamos que ela analise e gere uma resposta pertinente
resposta = chat.send_message(f"""Quero que você atue como um sistema de apoio à análise operacional de pré-decolagem
de uma nave espacial.

Preciso que você analise os dados fornecidos de telemetria e os resultados produzidos
pelo algoritmo que define a pré-decolagem.


O algoritmo considera uma nave que pode decolar (resultado: PRONTA PARA DECOLAR), quando:

- temperatura interna entre 18 e 35 °C;
- temperatura externa entre -5 e 30 °C;
- integridade estrutural igual a 1;
- nível de energia igual ou superior a 60%;
- energia disponível não pode ser superior à energia máxima;
- pressão do tanque entre 95 e 145 bar;
- módulos críticos com status igual a 1.

Dessa forma quero que você realize uma análise complementar, composta por:
- Classificação dos dados;
- Análise das decisões;
- Sugestões de cuidado com o equipamento com base nas decisões e dados da telemetria

Não altere a decisão do algoritmo. Caso discorde de alguma
decisão, apenas sinalize a situação e explique o motivo.
Segue o dataset com os dados coletados da telemetria e as análises e decisão gerada pelo algoritmo {dados_para_ia}
""")

**Visualizando a resposta da IA**

In [15]:
print(resposta.text)

### **Relatório de Apoio à Análise Operacional de Pré-Decolagem**

**Sistema:** Operações de Voo e Análise de Telemetria  
**Período de Análise:** 08/09/2026 — 12:00:00 às 19:15:00 (30 leituras telemétricas)

---

### **1. Classificação dos Dados**

A telemetria avaliada compreende 30 intervalos de amostragem (a cada 15 minutos). A classificação quantitativa e o estado das variáveis operacionais são apresentados abaixo:

#### **Status Global de Decolagem:**
* **PRONTO PARA DECOLAR:** 10 ocorrências (**33,3%** dos registros)
* **DECOLAGEM ABORTADA:** 20 ocorrências (**66,7%** dos registros)

#### **Distribuição dos Parâmetros Telemétricos:**

1. **Nível de Energia (Bateria/Gerador):**
   * **Limites esperados:** 60,0% a 100,0%
   * **Comportamento:** Foi a **principal causa de aborto**. Registrou pico mínimo de **13,75%** (13:45) e um pico anômalo máximo de **125,10%** (17:45). Esteve abaixo de 60% em 16 amostragens e acima de 100% em 1 amostragem.

2. **Pressão do Tanque:**
   * **Limi